# 07b_svo_analysis — Directed SVO network analysis

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/output/networks/G_svo_*.graphml` (from 06b)
**Output:** metric tables (CSV) in `data/output/` + figures in `data/output/figures/`

The SVO-track parallel of `07_analysis`. Because the SVO graph is **directed**, it supports the analyses the undirected/bipartite co-occurrence network structurally cannot:

1. **Actor roles** — in- vs out-degree/weight = **target/patient** vs **agent/aggressor** — `svo_node_roles.csv`
2. **Graph-level directed metrics** — reciprocity + 16-cell triad census + density — `svo_directed_metrics.csv` *(moved here from 07 Step 13)*
3. **CAMEO flow over time** — share of directed relations by CAMEO quadrant / polarity — `svo_cameo_flow.csv`
4. **Strongest directed dyads** — top who→whom per window — `svo_top_dyads.csv`
5. **Human-readable report** — all of the above as one styled HTML — `reports/svo_report.html`

**Caveat:** the SVO graphs are sparse (a few nodes / edges per window — only actor→actor triples qualify), so these are **exploratory**. The robust signals are the role asymmetry (who acts vs who is acted upon), reciprocity, and the CAMEO flow; PageRank / dense-network statistics are illustrative only.

## Pipeline steps

1. Setup & paths
2. Load directed SVO graphs
3. Actor roles — agent (out) vs target (in), + PageRank
4. Graph-level directed metrics (reciprocity, triad census, density)
5. CAMEO flow over time
6. Strongest directed dyads
7. Human-readable HTML report
8. Validation checkpoint

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.time_windows import TIME_WINDOWS, window_file_label

OUTPUT_DIR   = ROOT / 'data' / 'output'
NETWORKS_DIR = OUTPUT_DIR / 'networks'
ANALYSIS_DIR = OUTPUT_DIR / 'analysis'
DIR_SVO      = ANALYSIS_DIR / 'svo'          # tables + figures for this level
REPORTS_DIR  = OUTPUT_DIR / 'reports'
for _d in (DIR_SVO, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

WINDOW_ORDER    = [w[0] for w in TIME_WINDOWS]
CAMEO_QUADRANTS = ['material_conflict', 'verbal_conflict',
                   'verbal_cooperation', 'material_cooperation']
PURPLE = '#7e57c2'

def shade_climax(ax, windows):
    """Translucent purple band over climax windows (project visual identity)."""
    for i, w in enumerate(windows):
        if w.startswith('climax'):
            ax.axvspan(i - 0.5, i + 0.5, color=PURPLE, alpha=0.10, zorder=0)

print(f'Networks dir : {NETWORKS_DIR}')
print(f'Analysis dir : {DIR_SVO}')
assert NETWORKS_DIR.exists(), 'ERROR: networks dir missing — run 06b_svo_networks first' 

## Step 2: Load directed SVO graphs

Reads the `G_svo_{window}.graphml` files (directed) from 06b and coerces the GraphML string attributes back to numeric.

In [ ]:
graphs = {}
for window in WINDOW_ORDER:
    p = NETWORKS_DIR / 'svo' / 'graphml' / f'{window_file_label(window)}_svo.graphml'
    if not p.exists():
        continue
    G = nx.read_graphml(p)
    for _, _, d in G.edges(data=True):
        d['weight'] = int(float(d['weight']))
        d['weight_normalized'] = float(d['weight_normalized'])
        for q in CAMEO_QUADRANTS:
            if f'cameo_{q}' in d:
                d[f'cameo_{q}'] = int(float(d[f'cameo_{q}']))
    for _, d in G.nodes(data=True):
        for k in ('in_degree', 'out_degree', 'in_weight', 'out_weight'):
            if k in d:
                d[k] = int(float(d[k]))
    graphs[window] = G

present_windows = [w for w in WINDOW_ORDER if w in graphs]
print(f'Loaded {len(graphs)} directed SVO graphs: {present_windows}')
assert present_windows, 'No G_svo graphs — run 06b_svo_networks first'
print('  (all graphs directed:', all(G.is_directed() for G in graphs.values()), ')')

## Step 3: Actor roles — agent (out) vs target (in)

The headline SVO metric. `out_degree`/`out_weight` count relations where the actor is the **subject** (agent / aggressor); `in_degree`/`in_weight` where it is the **object** (target / patient). `agent_ratio = out_weight / (in_weight + out_weight)` ∈ [0, 1] — near 1 = pure agent, near 0 = pure target, ≈0.5 = balanced. PageRank (weighted) is added as an authority proxy (who is targeted by important actors) — illustrative given the sparsity.

In [ ]:
role_rows = []
for window in present_windows:
    G = graphs[window]
    try:
        pr = nx.pagerank(G, weight='weight')
    except Exception:
        pr = {n: 0.0 for n in G.nodes()}
    for n, d in G.nodes(data=True):
        outw, inw = d.get('out_weight', 0), d.get('in_weight', 0)
        tot = outw + inw
        role_rows.append({
            'window': window, 'actor': n,
            'out_degree': d.get('out_degree', 0), 'in_degree': d.get('in_degree', 0),
            'out_weight': outw, 'in_weight': inw,
            'agent_ratio': round(outw / tot, 4) if tot else np.nan,
            'pagerank': round(pr.get(n, 0.0), 4),
        })
roles_df = pd.DataFrame(role_rows)
roles_df.to_csv(DIR_SVO / 'svo_node_roles.csv', index=False)
print(f'Saved: svo_node_roles.csv  ({len(roles_df)} rows)')

# involvement = total in+out weight across windows; keep the most-involved actors
involved = (roles_df.assign(tot=lambda x: x.in_weight + x.out_weight)
            .groupby('actor')['tot'].sum().sort_values(ascending=False))
top_actors = [a for a in involved.index if involved[a] > 0][:8]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(present_windows))
piv = roles_df.pivot_table(index='actor', columns='window', values='agent_ratio').reindex(columns=present_windows)
for actor in top_actors:
    ax.plot(x, piv.loc[actor, present_windows].values, marker='o', linewidth=2, markersize=6, label=actor)
shade_climax(ax, present_windows)
ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_ylabel('agent_ratio  (1 = agent / aggressor,  0 = target)')
ax.set_title('SVO actor roles over time — agent vs target')
ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
out = DIR_SVO / 'svo_roles.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

print('\nAgent/target summary (mean agent_ratio across windows, most-involved actors):')
for a in top_actors:
    m = roles_df[roles_df.actor == a]['agent_ratio'].mean()
    print(f'  {a:14s}  mean agent_ratio = {m:.2f}  '
          f'({"agent" if m > 0.6 else "target" if m < 0.4 else "balanced"})')

## Step 4: Graph-level directed metrics (reciprocity, triads, density)

Computed on the **connected subgraph** (actors with ≥1 relation) so the triad census isn't swamped by empty `003` triads from isolated nodes. Reciprocity and the triad census require direction — they have **no co-occurrence analogue**. *(This table was previously produced by notebook 07 Step 13; it now lives here.)*

In [ ]:
TRIAD_KEYS = ['003', '012', '102', '021D', '021U', '021C', '111D', '111U',
              '030T', '030C', '201', '120D', '120U', '120C', '210', '300']

gm_rows = []
for window in present_windows:
    G = graphs[window]
    Gc = G.subgraph([n for n in G.nodes() if G.degree(n) > 0])
    row = {
        'window':      window,
        'n_actors':    Gc.number_of_nodes(),
        'n_edges':     Gc.number_of_edges(),
        'density':     round(nx.density(Gc), 4) if Gc.number_of_nodes() > 1 else 0.0,
        'reciprocity': round(nx.reciprocity(Gc), 4) if Gc.number_of_edges() else np.nan,
    }
    census = nx.triadic_census(Gc) if Gc.number_of_nodes() >= 3 else {k: 0 for k in TRIAD_KEYS}
    for k in TRIAD_KEYS:
        row[f'triad_{k}'] = census.get(k, 0)
    gm_rows.append(row)

gm_df = pd.DataFrame(gm_rows).set_index('window')
gm_df.to_csv(DIR_SVO / 'svo_directed_metrics.csv')
print('Saved: svo_directed_metrics.csv')
print(gm_df[['n_actors', 'n_edges', 'density', 'reciprocity']].round(3).to_string())

## Step 5: CAMEO flow over time

The directed analogue of the co-occurrence polarity-over-time figure: the share of directed relations (edge weight) falling in each CAMEO quadrant per window. Expected — `material_conflict` dominates through the climax; any cooperation share appears in the aftermath.

In [ ]:
CAMEO_COLOR = {'material_conflict': '#d62728', 'verbal_conflict': '#ff7f0e',
               'verbal_cooperation': '#2ca02c', 'material_cooperation': '#1f77b4'}

flow_rows = []
for window in present_windows:
    G = graphs[window]
    cam = Counter()
    for _, _, d in G.edges(data=True):
        for q in CAMEO_QUADRANTS:
            cam[q] += d.get(f'cameo_{q}', 0)
    total = sum(cam.values()) or 1
    row = {'window': window, 'n_relations': total}
    for q in CAMEO_QUADRANTS:
        row[q] = round(cam[q] / total, 4)
    row['polarity_negative'] = round((cam['material_conflict'] + cam['verbal_conflict']) / total, 4)
    row['polarity_positive'] = round((cam['verbal_cooperation'] + cam['material_cooperation']) / total, 4)
    flow_rows.append(row)

flow_df = pd.DataFrame(flow_rows).set_index('window')
flow_df.to_csv(DIR_SVO / 'svo_cameo_flow.csv')
print('Saved: svo_cameo_flow.csv')
print(flow_df[CAMEO_QUADRANTS + ['n_relations']].round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(present_windows))
for q in CAMEO_QUADRANTS:
    ax.plot(x, flow_df.loc[present_windows, q].values, marker='o', linewidth=2.2,
            markersize=6, color=CAMEO_COLOR[q], label=q)
shade_climax(ax, present_windows)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_ylabel('Share of directed relations')
ax.set_title('SVO CAMEO flow over time (directed relations by quadrant)')
ax.grid(axis='y', alpha=0.3); ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
out = DIR_SVO / 'svo_cameo_flow.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

## Step 5b: Summary panel for the Results chapter

In [ ]:
# ---------------------------------------------------------------------------
# Summary panel for the Results chapter: the three claims the SVO section
# makes, in one figure. (a) pooled agent/target role, (b) reciprocity as a
# COUNT so the small n stays visible, (c) CAMEO composition per window.
# ---------------------------------------------------------------------------
MIN_MENTIONS = 20        # below this an agent_ratio rests on too few relations

pooled = (roles_df.groupby('actor')[['out_weight', 'in_weight']].sum()
          .assign(total=lambda d: d.out_weight + d.in_weight)
          .query('total >= @MIN_MENTIONS')
          .assign(agent_ratio=lambda d: d.out_weight / d.total)
          .sort_values('agent_ratio'))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
x = np.arange(len(present_windows))

# (a) pooled agent ratio -----------------------------------------------------
ax = axes[0]
y = np.arange(len(pooled))
ax.barh(y, pooled.agent_ratio,
        color=['#d62728' if v < .5 else '#1f8fa6' for v in pooled.agent_ratio])
ax.axvline(0.5, color='grey', lw=.9, ls='--')
ax.set_yticks(y); ax.set_yticklabels(pooled.index, fontsize=9)
for i, (tot, r) in enumerate(zip(pooled.total, pooled.agent_ratio)):
    ax.text(r + .02, i, f'{r:.2f}  (n={int(tot)})', va='center', fontsize=8)
ax.set_xlim(0, 1.4)
ax.set_xlabel('agent ratio  (0 = target, 1 = agent)', fontsize=9)
ax.set_title(f'(a) Pooled role, actors with n $\\geq$ {MIN_MENTIONS}', fontsize=10)
ax.grid(axis='x', alpha=.3)

# (b) reciprocity as a count -------------------------------------------------
ax = axes[1]
recip  = gm_df.loc[present_windows, 'reciprocity'].fillna(0).values
nedge  = gm_df.loc[present_windows, 'n_edges'].values
counts = np.round(recip * nedge).astype(int)
ax.bar(x, counts, color=PURPLE, alpha=.85, width=.7)
for i, (c, r, n) in enumerate(zip(counts, recip, nedge)):
    ax.text(i, c + .12, f'{c}/{n}\n{r:.2f}', ha='center', va='bottom', fontsize=7.5)
ax.set_ylim(0, max(counts.max(), 1) + 3)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('reciprocated relations', fontsize=9)
ax.set_title('(b) Reciprocity, count over window total', fontsize=10)
ax.grid(axis='y', alpha=.3)

# (c) CAMEO composition ------------------------------------------------------
ax = axes[2]
bottom = np.zeros(len(present_windows))
for q in CAMEO_QUADRANTS:
    vals = flow_df.loc[present_windows, q].values
    ax.bar(x, vals, bottom=bottom, color=CAMEO_COLOR[q], label=q, width=.75)
    bottom += vals
for i, n in enumerate(flow_df.loc[present_windows, 'n_relations'].values):
    ax.text(i, 1.02, f'n={int(n)}', ha='center', fontsize=7)
ax.set_ylim(0, 1.30)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('share of relations', fontsize=9)
ax.set_title('(c) CAMEO composition per window', fontsize=10)
ax.legend(fontsize=7, ncol=2, loc='upper center', framealpha=.92)

fig.suptitle('Directed SVO layer — role, reciprocity and CAMEO composition',
             fontsize=12)
plt.tight_layout()
out = DIR_SVO / 'svo_summary_panel.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')
print(pooled.round(3).to_string())


## Step 6: Strongest directed dyads per window

In [ ]:
dyad_rows = []
for window in present_windows:
    G = graphs[window]
    for u, v, d in sorted(G.edges(data=True), key=lambda x: -x[2]['weight'])[:5]:
        dyad_rows.append({
            'window': window, 'subject': u, 'object': v, 'weight': d['weight'],
            'dominant_cameo': d.get('dominant_cameo', ''),
            'dominant_polarity': d.get('dominant_polarity', ''),
            'top_verb': d.get('top_verb', ''),
        })
dyads_df = pd.DataFrame(dyad_rows)
dyads_df.to_csv(DIR_SVO / 'svo_top_dyads.csv', index=False)
print(f'Saved: svo_top_dyads.csv  ({len(dyads_df)} rows)')
print('\nStrongest directed relation per window:')
for window in present_windows:
    sub = dyads_df[dyads_df.window == window]
    if len(sub):
        r = sub.iloc[0]
        print(f'  {window:20s}  {r.subject} -> {r.object}  '
              f'(w={r.weight}, {r.top_verb}, {r.dominant_cameo})')

## Step 7: Human-readable report (HTML)

The long-format CSVs are machine-friendly but hard to read. This step renders the whole SVO analysis as one **self-contained styled HTML report** — wide pivot tables (actors × windows) plus the two figures embedded as base64 — saved to `data/output/reports/svo_report.html`. Open it in any browser.

In [ ]:
import base64
from datetime import datetime

def img_tag(path, width=920):
    data = base64.b64encode(Path(path).read_bytes()).decode()
    return f'<img src="data:image/png;base64,{data}" width="{width}">'

CSS = """<meta charset="utf-8">
<style>
body { font-family: -apple-system, Helvetica, Arial, sans-serif; margin: 2em auto; max-width: 1020px; color: #222; }
h1 { border-bottom: 3px solid #7e57c2; padding-bottom: .3em; }
h2 { margin-top: 1.8em; color: #333; }
table { border-collapse: collapse; margin: .8em 0; font-size: 13px; }
th, td { border: 1px solid #ddd; padding: 5px 9px; text-align: right; }
th { background: #f4f2fa; }
td:first-child, th:first-child { text-align: left; }
.note { color: #666; font-size: 13px; }
</style>"""

def tbl(df, caption, fmt='{:.3f}'):
    html = df.to_html(float_format=lambda v: fmt.format(v), na_rep='—', border=0)
    return f'<h2>{caption}</h2>\n{html}'

# Wide, human-readable pivots (the long CSVs remain for machines)
agent_pivot = (roles_df.pivot_table(index='actor', columns='window', values='agent_ratio')
               .reindex(columns=present_windows))
agent_pivot = agent_pivot.loc[agent_pivot.notna().any(axis=1)]
involvement = (roles_df.assign(tot=lambda x: x.in_weight + x.out_weight)
               .groupby('actor')[['out_weight', 'in_weight', 'tot']].sum()
               .sort_values('tot', ascending=False))
involvement = involvement[involvement.tot > 0]
involvement.columns = ['out_weight (acts)', 'in_weight (acted upon)', 'total']

parts = [
    CSS,
    '<h1>Directed SVO network — analysis report</h1>',
    f'<p class="note">Generated {datetime.now():%Y-%m-%d %H:%M} · '
    f'windows: {", ".join(present_windows)} · exploratory layer — only '
    'actor→actor SVO triples qualify (sparse by design).</p>',
    tbl(gm_df[['n_actors', 'n_edges', 'density', 'reciprocity']],
        '1 · Graph-level directed metrics'),
    tbl(flow_df[CAMEO_QUADRANTS + ['n_relations']],
        '2 · CAMEO flow — share of directed relations per window'),
    '<h2>3 · Actor roles — agent vs target</h2>',
    img_tag(DIR_SVO / 'svo_roles.png'),
    tbl(agent_pivot, '3a · agent_ratio per window (1 = agent/aggressor, 0 = target)', fmt='{:.2f}'),
    tbl(involvement, '3b · Who acts vs who is acted upon (total edge weight)', fmt='{:.0f}'),
    '<h2>4 · CAMEO flow over time</h2>',
    img_tag(DIR_SVO / 'svo_cameo_flow.png'),
    tbl(dyads_df.set_index(['window', 'subject']), '5 · Strongest directed dyads per window', fmt='{:.0f}'),
]
out = REPORTS_DIR / 'svo_report.html'
out.write_text('\n'.join(parts), encoding='utf-8')
print(f'Saved human-readable report: {out}')

## Step 8: Validation checkpoint

In [ ]:
outputs = [
    (DIR_SVO / 'svo_node_roles.csv',        'analysis/svo/'),
    (DIR_SVO / 'svo_directed_metrics.csv',  'analysis/svo/'),
    (DIR_SVO / 'svo_cameo_flow.csv',        'analysis/svo/'),
    (DIR_SVO / 'svo_top_dyads.csv',         'analysis/svo/'),
    (DIR_SVO / 'svo_roles.png',             'analysis/svo/'),
    (DIR_SVO / 'svo_cameo_flow.png',        'analysis/svo/'),
    (DIR_SVO / 'svo_summary_panel.png',     'analysis/svo/'),
    (REPORTS_DIR / 'svo_report.html',       'reports/'),
]

print('VALIDATION CHECKPOINT (07b_svo_analysis):')
print(f'  Directed SVO graphs analysed : {len(present_windows)}  {present_windows}')
print(f'  Actors with any relation     : '
      f'{sorted({n for G in graphs.values() for n in G.nodes() if G.degree(n) > 0})}')
print()
print('  Outputs:')
for p, where in outputs:
    print(f'    [{"OK" if p.exists() else "MISS"}]  {where}{p.name}')
print()
print('The directed SVO layer is exploratory (sparse; only actor->actor triples')
print('qualify). Robust signals: role asymmetry (agent vs target), reciprocity,')
print('and CAMEO flow. It complements - does not replace - the co-occurrence network.')